In [0]:

%sql
CREATE CATALOG IF NOT EXISTS retail_ai3;
CREATE SCHEMA IF NOT EXISTS retail_ai3.bronze;
CREATE SCHEMA IF NOT EXISTS retail_ai3.silver;
CREATE SCHEMA IF NOT EXISTS retail_ai3.gold;

-- GenAI assets
CREATE SCHEMA IF NOT EXISTS retail_ai3.ai;


In [0]:
path = "/Volumes/retail_ai3/bronze/raw/generated_customers.csv"

df_customers = (spark.read
      .format("csv")
      .option("header", True)
      .option("inferSchema", True)
      .load(path))

display(df_customers)


In [0]:
path = "/Volumes/retail_ai3/bronze/raw/orders.csv"

df_orders = (spark.read
      .format("csv")
      .option("header", True)
      .option("inferSchema", True)
      .load(path))

display(df_orders)


In [0]:
path = "/Volumes/retail_ai3/bronze/raw/products_docs.json"

df_products = spark.read.json(path)

display(df_products.orderBy("product_name"))

In [0]:
path = "/Volumes/retail_ai3/bronze/raw/customer_service_data.csv"

df_customer_service_request = (spark.read
      .format("csv")
      .option("header", True)
      .option("inferSchema", True)
      .load(path))

display(df_customer_service_request)


In [0]:
path = "/Volumes/retail_ai3/bronze/raw/company_policy.csv"

df_company_policy = (spark.read
      .format("csv")
      .option("header", True)
      .option("inferSchema", True)
      .load(path))

display(df_company_policy)


In [0]:
df_customers.write.format("delta").mode("overwrite").saveAsTable(
    "retail_ai3.bronze.customers_raw"
)
df_products.write.format("delta").mode("overwrite").saveAsTable(
    "retail_ai3.bronze.products_raw"
)
df_customer_service_request.write.format("delta").mode("overwrite").saveAsTable(
    "retail_ai3.bronze.customer_service_request_raw"
)
df_orders.write.format("delta").mode("overwrite").saveAsTable(
    "retail_ai3.bronze.orders_raw"
)
df_company_policy.write.format("delta").mode("overwrite").saveAsTable(
    "retail_ai3.bronze.company_policy_raw"
)
display(spark.sql("DESCRIBE DETAIL retail_ai3.bronze.customers_raw"))
display(spark.sql("DESCRIBE DETAIL retail_ai3.bronze.products_raw"))
display(spark.sql("DESCRIBE DETAIL retail_ai3.bronze.customer_service_request_raw"))
display(spark.sql("DESCRIBE DETAIL retail_ai3.bronze.orders_raw"))
display(spark.sql("DESCRIBE DETAIL retail_ai3.bronze.company_policy_raw"))


In [0]:
from pyspark.sql import functions as F

src = "retail_ai3.bronze.customer_service_request_raw"
tgt = "retail_ai3.silver.customer_service_request"

df = spark.table(src)

# 1️⃣ Trim all string columns
for col_name, dtype in df.dtypes:
    if dtype == "string":
        df = df.withColumn(col_name, F.trim(F.col(col_name)))

# 2️⃣ Standardize issue_category (uppercase)
if "issue_category" in df.columns:
    df = df.withColumn("issue_category", F.upper(F.col("issue_category")))

# 3️⃣ Standardize status (InitCap format)
if "status" in df.columns:
    df = df.withColumn("status", F.initcap(F.col("status")))

# 4️⃣ Clean phone number (remove non-digits)
if "phone" in df.columns:
    df = df.withColumn("phone", F.regexp_replace(F.col("phone"), r"\D", ""))

# 5️⃣ Convert created_date to date type (if exists)
if "created_date" in df.columns:
    df = df.withColumn("created_date", F.to_date(F.col("created_date")))

# 6️⃣ Add ingestion timestamp
df = df.withColumn("silver_ingestion_ts", F.current_timestamp())

silver_cs = df

(silver_cs.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(tgt))

print("✅ Silver created with simple transformations:", tgt)


In [0]:
from pyspark.sql import functions as F

src = "retail_ai3.bronze.company_policy_raw"
tgt = "retail_ai3.silver.company_policy"

df = spark.table(src)

# Trim all string columns
for col_name, dtype in df.dtypes:
    if dtype == "string":
        df = df.withColumn(col_name, F.trim(F.col(col_name)))

silver_policy = df

(silver_policy.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(tgt))

print("Silver created with ALL columns:", tgt)


In [0]:
#create silver

from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.table("retail_ai3.bronze.customers_raw")

w = Window.partitionBy("customer_id").orderBy("customer_id")

silver_customers = (
    df
    .withColumn("customer_id", F.col("customer_id").cast("string"))
    .withColumn("email", F.lower(F.col("email")))
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .drop("rn")
    .filter(F.col("email").contains("@"))
)

(silver_customers.write.format("delta")
 .mode("overwrite")
 .saveAsTable("retail_ai3.silver.customers"))



In [0]:
df = spark.table("retail_ai3.bronze.products_raw")

silver_products = (
    df
    .withColumn("product_id", F.col("product_id").cast("string"))
    .withColumn("price", F.col("price").cast("double"))
)

(silver_products.write.format("delta")
 .mode("overwrite")
 .saveAsTable("retail_ai3.silver.products"))

In [0]:

orders = (spark.table("retail_ai3.bronze.orders_raw")
          .withColumn("order_id", F.col("order_id").cast("string"))
          .withColumn("customer_id", F.col("customer_id").cast("string"))
          .withColumn("product_id", F.col("product_id").cast("string"))
          .withColumn("quantity", F.col("quantity").cast("int"))
          .withColumn("order_date", F.col("order_date").cast("timestamp")))

products = spark.table("retail_ai3.silver.products").select("product_id","price")

silver_orders = (
    orders.join(products, "product_id", "left")
    .withColumn("amount", F.col("quantity") * F.col("price"))
)

(silver_orders.write.format("delta")
 .mode("overwrite")
 .saveAsTable("retail_ai3.silver.orders"))


In [0]:
%sql
--gold layer
CREATE OR REPLACE TABLE retail_ai3.gold.product_wise_revenue AS
SELECT
  p.product_name,
  round(sum(amount), 2) AS revenue
FROM retail_ai3.silver.orders o
JOIN retail_ai3.silver.products p
  ON o.product_id = p.product_id
GROUP BY p.product_name;


In [0]:
%sql
select * from retail_ai3.gold.product_wise_revenue

In [0]:
%sql
CREATE OR REPLACE TABLE retail_ai3.gold.top_customers AS
SELECT
  o.customer_id,
  c.name,
  round(sum(o.amount), 2) AS total_spent
FROM retail_ai3.silver.orders o
JOIN retail_ai3.silver.customers c
  ON o.customer_id = c.customer_id
GROUP BY 1,2
ORDER BY total_spent DESC
LIMIT 20;


In [0]:
%sql
select * from retail_ai3.gold.top_customers ;

In [0]:
from pyspark.sql import functions as F

src = "retail_ai3.silver.customer_service_request"
tgt = "retail_ai3.gold.customer_service_summary"

df = spark.table(src)

gold_cs = (
    df.groupBy("issue_category")
      .agg(F.count("*").alias("total_records"))
)

(gold_cs.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable(tgt))

print("Gold summary created:", tgt)


In [0]:
%sql
select * from retail_ai3.gold.customer_service_summary;

In [0]:
%sql
CREATE OR REPLACE FUNCTION
  IDENTIFIER('retail_ai3.ai.get_latest_return')()
RETURNS TABLE(purchase_date DATE, issue_category STRING, issue_description STRING, name STRING)
COMMENT 'Returns the most recent customer service interaction, such as returns.'
RETURN (
  SELECT
    CAST(date_time AS DATE) AS purchase_date,
    issue_category,
    issue_description,
    name
  FROM retail_ai3.silver.customer_service_request
  ORDER BY date_time DESC
  LIMIT 1
);

In [0]:
%sql
select * from retail_ai3.silver.company_policy;

In [0]:
%sql
CREATE OR REPLACE FUNCTION
  IDENTIFIER('retail_ai3.ai.get_return_policy')()
RETURNS TABLE (
  policy           STRING,
  policy_details   STRING,
  last_updated     DATE
)
COMMENT 'Returns the details of the Return Policy'
LANGUAGE SQL
RETURN (
  SELECT
    policy,
    policy_details,
    last_updated
  FROM retail_ai3.silver.company_policy
  WHERE policy = 'Return Policy'
  LIMIT 1
);

In [0]:
%sql
select * from IDENTIFIER('retail_ai3.ai.get_return_policy')()

In [0]:
%sql
CREATE OR REPLACE FUNCTION
  IDENTIFIER('retail_ai3.ai.get_order_history')(user_name STRING)
RETURNS TABLE (returns_last_12_months INT, issue_category STRING, todays_date DATE)
COMMENT 'This takes the user_name of a customer as an input and returns the number of returns and the issue category'
LANGUAGE SQL
RETURN
SELECT count(*) as returns_last_12_months, issue_category, now() as todays_date
FROM retail_ai3.silver.customer_service_request
WHERE name = name
GROUP BY issue_category;

In [0]:
%sql
select * from IDENTIFIER('retail_ai3.ai.get_order_history')('Nicolas Pelaez')

In [0]:
def get_todays_date() -> str:
    """
    Returns today's date in 'YYYY-MM-DD' format.

    Returns:
        str: Today's date in 'YYYY-MM-DD' format.
    """
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d")

In [0]:
%pip install -U -qqqq backoff databricks-openai uv databricks-agents mlflow-skinny[databricks]
dbutils.library.restartPython()

In [0]:
%pip install databricks-sdk


In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

group_names = ["data_engineer", "data_analyst", "gov_auditor_reader", "business_user"]

created = {}
for name in group_names:
    # try create; if exists, fetch
    try:
        g = w.groups.create(display_name=name)
        created[name] = g.id
        print("Created group:", name, g.id)
    except Exception:
        # fallback: find existing
        gs = w.groups.list(filter=f'displayName eq "{name}"')
        g = next(iter(gs), None)
        if g:
            created[name] = g.id
            print("Group already exists:", name, g.id)
        else:
            raise


In [0]:
demo_users = [
    ("sourabh_dataeng@azurelib.com", "DE"),
    ("an1@azurelib.com", "Analyst"),
    ("gov_auditor@azurelib.com", "Auditor "),
    ("biz1@azurelib.com", "Biz"),
]

user_ids = {}
for email, name in demo_users:
    try:
        u = w.users.create(user_name=email, display_name=name)
        user_ids[email] = u.id
        print("Created user:", email, u.id)
    except Exception:
        # user may already exist, lookup by filter
        users = w.users.list(filter=f'userName eq "{email}"')
        u = next(iter(users), None)
        if u:
            user_ids[email] = u.id
            print("User already exists:", email, u.id)
        else:
            raise


In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

def add_member_to_group(group_id: str, user_id: str):
    w.api_client.do(
        method="PATCH",
        path=f"/api/2.0/preview/scim/v2/Groups/{group_id}",
        body={
            "schemas": ["urn:ietf:params:scim:api:messages:2.0:PatchOp"],
            "Operations": [
                {
                    "op": "Add",
                    "path": "members",
                    "value": [{"value": user_id}]
                }
            ]
        }
    )
    print(f" Added user {user_id} to group {group_id}")


In [0]:
add_member_to_group(created["data_engineer"], user_ids["sourabh_dataeng@azurelib.com"])
add_member_to_group(created["data_analyst"], user_ids["an1@azurelib.com"])
add_member_to_group(created["gov_auditor_reader"], user_ids["gov_auditor@azurelib.com"])
add_member_to_group(created["business_user"], user_ids["biz1@azurelib.com"])


In [0]:
%sql
CREATE OR REPLACE VIEW retail_ai3.silver.customers_masked AS
SELECT
  customer_id, name, address,
  CASE WHEN is_account_group_member('gov_auditor_reader') THEN email
       ELSE regexp_replace(email, '(^.).*(@.*$)', '$1***$2') END AS email

FROM retail_ai3.silver.customers;


In [0]:
%sql
select * from retail_ai3.silver.customers_masked;

In [0]:
%sql
select * from retail_ai3.silver.products;

In [0]:
SOURCE_TABLE='retail_ai3.silver.products'
spark.sql(f"ALTER TABLE {SOURCE_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print("✅ CDF enabled on:", SOURCE_TABLE)


In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import vectorsearch as vs

w = WorkspaceClient()

SOURCE_TABLE = "retail_ai3.silver.products"
ENDPOINT_NAME = "retail-vs-endpoint"

CATALOG = "retail_ai3"
SCHEMA = "ai"
INDEX_FULL_NAME = f"{CATALOG}.{SCHEMA}.products_index"

EMBED_TEXT_COL = "product_document"          # best column for RAG
EMBEDDING_ENDPOINT = "databricks-bge-large-en"  # must exist in your workspace


In [0]:
spark.sql(f"ALTER TABLE {SOURCE_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print("✅ CDF enabled:", SOURCE_TABLE)


In [0]:
[w.name for w in WorkspaceClient().serving_endpoints.list()]


In [0]:
existing = [e.name for e in w.vector_search_endpoints.list_endpoints()]
if ENDPOINT_NAME in existing:
    print("✅ Endpoint already exists:", ENDPOINT_NAME)
else:
    w.vector_search_endpoints.create_endpoint(
        name=ENDPOINT_NAME,
        endpoint_type=vs.EndpointType.STANDARD
    )
    print("✅ Endpoint creation requested:", ENDPOINT_NAME)


In [0]:
# If index exists, delete (optional)
try:
    w.vector_search_indexes.get_index(INDEX_FULL_NAME)
    print("ℹ️ Index exists, deleting:", INDEX_FULL_NAME)
    w.vector_search_indexes.delete_index(INDEX_FULL_NAME)
except Exception:
    pass

spec = vs.DeltaSyncVectorIndexSpecRequest(
    source_table=SOURCE_TABLE,
    pipeline_type=vs.PipelineType.TRIGGERED,   # TRIGGERED is good for demo / beginner
    embedding_source_columns=[
        vs.EmbeddingSourceColumn(
            name=EMBED_TEXT_COL,
            embedding_model_endpoint_name=EMBEDDING_ENDPOINT
        )
    ],
    columns_to_sync=[
        "product_id", "product_name", "price",
        "product_description", "product_document"
    ]
)

resp = w.vector_search_indexes.create_index(
    name=INDEX_FULL_NAME,
    endpoint_name=ENDPOINT_NAME,
    primary_key="product_id",
    index_type=vs.VectorIndexType.DELTA_SYNC,
    delta_sync_index_spec=spec
)

print("✅ Index create submitted:", resp)
print("✅ Index:", INDEX_FULL_NAME)


In [0]:
query = "winter wears"

res = w.vector_search_indexes.query_index(
    index_name=INDEX_FULL_NAME,
    query_text=query,
    num_results=3,
    columns=["product_id", "product_name", "price", "product_description"]
)

res


In [0]:
cols = ["product_id", "product_name", "price", "product_description"]
rows = res.result.data_array

out_df = spark.createDataFrame([dict(zip(cols, r)) for r in rows])
display(out_df)
